In [1]:
%reload_ext autoreload
%autoreload 2

from MBN_Res_Constrn import MBN_RC
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [4]:
#put network you are using eg, full, node removed
name='RE'

#put itr as number of iterations you want
itr=600
#to get and store iterations data
c=400

#to store iteration data
df_itr=pd.DataFrame()   

#to store local order data
df_loc_data = pd.DataFrame()

#Lists to store transition time and time spent in sync state
trans_time=[]
state_time=[]


while c<itr:
    print("Current iteration is",c)
    mbn = MBN_RC(nepochs=40000, 
                 dt=0.05, 
                 lambda_o=2.86, 
                 alpha=0.01,
                 beta=0.002,
                 plot_bifurcation=False)
    
    mbn.run_model()

    #df to store 1 iteration data
    df=pd.DataFrame(mbn.GLOBAL_ORDER_VERBOSE)
    #concatenating each iteration as a column
    df_itr=pd.concat([df_itr,df], axis=1)

    #changing headers to number of iterations
    df_itr.columns=range(0,df_itr.shape[1])
    #storing dataframe in csv file
    df_itr.to_csv("data4.csv")

    #smoothing the dataframe
    df_smooth=df.rolling(window=800, center=True).mean()

    #defining time and timesteps for data
    time_steps = list(range(0,df_itr.shape[0]))   
    time=np.multiply(time_steps,mbn.dt)

    #dropping the NaN values
    df_na = df_smooth.dropna()
    

    #defining time and time steps for df_na data
    #can drop these 2 lines
    time_steps_na = list(range(0,df_na.shape[0])) 
    time_na=np.multiply(time_steps_na,mbn.dt)

    
    data = np.array(df_na)
    #Looping one cycle of finding transition and time spent in sync state
    while True:
        
        index_arr=np.where(data >= 0.4)[0]

        #Finding if iteration has a transition
        if index_arr.size > 0:
            upper_crossing = index_arr[0]
           #finding transition time thresholds     
            l1 = np.where(data >= 0.1)[0]
            l2=np.where(data <= 0.101)[0]
            low_intersection = np.intersect1d(l1, l2)
            #making sure lower thresholds are for 1st transition
            low_upd =low_intersection[low_intersection<upper_crossing]

            lower_crossing=low_upd[-1]
        
            t=(upper_crossing-lower_crossing)*mbn.dt
            trans_time.append(t)
            #trans_df=pd.concat([trans_df, df_na[i]], axis=1)
            #trans_df stores the iterations which have transitions

            #Extracting local order data for above iteration
            df_dum = pd.DataFrame()
            #defining local order data timestep thresholds
        
            #checking where it crossed 0.3
            m_loc=np.where(data >= 0.3)[0][0]
        

        
            lt_loc = m_loc - 3000
            ut_loc = m_loc + 2000

            #extracting data from df_loc
            header_list=list(range(lt_loc,ut_loc))
            #filtering those columns which lie in between 0 to mbn.nepochs
            header_list_f = [x for x in header_list if 0 <= x < mbn.nepochs]
            df_dum = mbn.df_loc[header_list_f]
            #changing column numbers so that they be concated 1 below other
            df_dum.columns = range(0,df_dum.shape[1])
        

            #creating multi-index dataframe
            index=[[c]*426,list(range(0,426))]
            df_dum=df_dum.set_index(index)
            df_loc_data = pd.concat([df_loc_data,df_dum])
        

        
            #checking for another transition
            fwd_data=np.array(data[upper_crossing:])
        

            #upper_crs=np.where(fwd_data >= 0.45)[0][-1]
            check=np.where(fwd_data <= 0.2)[0]

            if check.size > 0:
                ind=np.where(fwd_data <= 0.1)[0]
                if ind.size > 0:

                    lower_crs = ind[0]
                    lower_crs_up = lower_crs+upper_crossing
                    st_time=(lower_crs_up-lower_crossing)*mbn.dt
                    state_time.append(st_time)
                    #creating data for other cycle of transition and state time
                    data= np.array(data[lower_crs_up:])
                #trans_2df=pd.concat([trans_2df, df_na[i]], axis=1)
                else:
                    
                    break
                    

            else:
                #continue
                
                break
                

        else:
                #data_no_trans=pd.concat([data_no_trans, df_na[i]], axis=1)
            
            break
    
    #increasing count by 1
    c+=1
 

    
#define network for which you are calculatingab

df_tt = pd.DataFrame(trans_time,columns=[name])
df_st=  pd.DataFrame(state_time,columns=[name])


print(np.array(trans_time))
print(np.array(state_time))
print(f'The number of iterations with transition is {len(trans_time)}')    
print(f'The number of iterations with more than 1 transition is {len(state_time)}') 


df_tt.to_csv('tt4_600.csv')
df_st.to_csv('st4_600.csv')
df_loc_data=df_loc_data.astype(np.float32)
df_loc_data.to_pickle('l600.bz2',compression='bz2')

Current iteration is 400
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/../Model/tES/tES_Adaptive.py:141: RuntimeWarning: divide by zero encountered in divide
  K_ = 1 / self.K


0.08944869507287811
Current iteration is 401
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/../Model/tES/tES_Adaptive.py:141: RuntimeWarning: divide by zero encountered in divide
  K_ = 1 / self.K


0.066411703034841
Current iteration is 402
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


...

0.39167790425357346
Current iteration is 598
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/../Model/tES/tES_Adaptive.py:141: RuntimeWarning: divide by zero encountered in divide
  K_ = 1 / self.K


0.06496316285464088
Current iteration is 599
/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/MouseBrainLib/mb_communities.npz already exists
DataUtils initialized


/home/jupyter-nimish/srg-tes-epilepsy-models/analysis/Mouse brain network/../Model/tES/tES_Adaptive.py:141: RuntimeWarning: divide by zero encountered in divide
  K_ = 1 / self.K


0.06174868935795736
[  66.35   33.45   28.4    36.6    48.45  101.55   28.     29.     33.8
   38.05 1015.3   114.3    84.8    48.5    29.75   60.35   62.35   63.45
   89.4    29.15   28.3    95.75   31.7    26.8   109.5    58.6    52.5
  316.4    55.45   33.4    37.25   88.4    41.8    36.2    30.25   83.5
  552.65  209.35   28.35   44.15  105.45   31.8   129.65   72.25   30.15
   27.95  508.65   97.85   60.1    65.25   98.4    63.5    32.6    35.4
   60.7   261.35   30.45   71.1    34.1   147.9    27.2   186.25   49.95
   34.35   29.9    45.2    29.05   27.4    31.65   31.4   228.8    33.85
  111.15   29.35   32.05   39.6    96.85   32.55   30.45   25.75   29.1
   44.     43.55   76.05   61.1    49.75  111.45   82.85   50.6    46.4
   30.6    29.     56.95   30.05   92.7   129.65   34.25   38.6    61.65
   32.7    41.35   27.5    42.     69.6    36.45   26.75   34.5    38.15
   40.4    52.25   35.65  209.75   34.75   33.65   27.1   186.1   102.6
   28.5 ]
[ 165.45  137.75 1041.55  68